In [1]:
from Run_Designs import design_experiments, design_initial_experiments
from methods.DataPrep import dataprep
import numpy as np
import pandas as pd

1. C - List of number of categroies corresponding to each categorical variable
2. Nc - Number of categorical variables
3. Nx - Continuous variables
4. h - categorical
5. x - continuous
6. z - [h, x]

7. nDim - Nc + Nx
8. bounds - Lower and Upper bounds of both Catergorical variables and continuous variables

9. n_iter - Number of iterations to run the algorithm
10. initN - Number of intial data points
11. batch_size (b) - Number of experiments to be generated in each iteration

12. acq_type - Acquisition Type
13. Meas_Noise - Measurement Noise in the target 
14. Trade-off : Explorarion exploitation balance

In [2]:
Stock_solid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_solid')
Stock_liquid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_liquid')

Carbon_Names = Stock_solid['Carbon Source'].values.tolist()
Carbon_Names.append(Stock_liquid['Carbon Source'][1])
Carbon_Names.append(Stock_liquid['Carbon Source'][2])

Carbon_Ub = [50]*17 #[g/L]
Carbon_Ub.append(10) # Glycerol [mL/L]
Carbon_Ub.append(50) #Ethanol [mL/L]

In [3]:
Mol_List = ['KPhaffii_HSA'] #, 'KPhaffii_RBDJ', 'Kphaffii_Trastuzumab'
Data_Molecules = {}
for m in Mol_List:
    Data_Molecules[m] = pd.read_excel("KPhaffii_Experiments.xlsx", m).iloc[:,1:]
    for i in range(Data_Molecules[m].shape[0]):
        Data_Molecules[m].iloc[i,0] = Carbon_Names.index(Data_Molecules[m].iloc[i,0])
        Data_Molecules[m].iloc[i,1] = Data_Molecules[m].iloc[i,1]/Carbon_Ub[Data_Molecules[m].iloc[i,0]]
        Data_Molecules[m].iloc[i,2:4] = Data_Molecules[m].iloc[i,2:4]/10
        
data_modified_np = {}
result_modified_np = {}
cnt_m = 0
for m in Mol_List:
    data_modified = []
    result_modified = []
    data_init = Data_Molecules[m].iloc[:,0:4].values.astype('float64')
    result_init = Data_Molecules[m]['SP [mg/L/OD600]'].values.reshape(-1,1).astype('float64')
    for nc in range(data_init.shape[0]):
#         if data_init[nc, 1] == 0:
#             temp = np.concatenate((np.arange(0, 19).reshape(-1, 1), np.zeros((19, 1)),
#                                    np.multiply(data_init[nc, 2:], np.ones((19, 2)))), axis=1)
#             temp_res = result_init[nc] * np.ones((19, 1))
#             data_modified + temp.tolist()
#             result_modified + temp_res.tolist()
#         else:
        data_modified.append(data_init[nc, :])
        result_modified.append(result_init[nc])
    
    temp1 = np.array(data_modified)
    data_modified_np[m] = np.concatenate((temp1, cnt_m*np.ones((temp1.shape[0], 1))), axis =1)
    result_modified_np[m] = np.array(result_modified).reshape(-1,1)
    
    cnt_m = cnt_m+1

In [4]:
initN = 23
Nc = 2
C_list = [2, 19] #
assert Nc == len(C_list)
Nx = 3
nDim = Nc + Nx
Niter = 1
batch_size = 11
bounds = [{'name': 'Carbon_Type', 'type': 'categorical', 'domain': np.arange(0, 19)}, #Carbon Source type
          {'name': 'Conc_Carbon', 'type': 'continuous', 'domain': (0, 1)},
            {'name': 'Gly_OG', 'type': 'continuous', 'domain': (0, 1)},
            {'name': 'Met_Prod', 'type': 'continuous', 'domain': (0, 1)}]

#0 - lb and 1 - ub
data_param = {'C': C_list, 'Nc': Nc, 'Nx': Nx, 'nDim': Nc+Nx, 'bounds': bounds, 'approach_type':'CoCa',
'prob_type': 'UnConstrained', 'initN': initN, 'trade_off': 3, 'Meas_Noise':0.1, 'batch_size': 11}

data_param_Coreg = data_param.copy()
data_param_Coreg['Nc'] = 1
data_param_Coreg['nDim'] = data_param_Coreg['Nc'] + data_param_Coreg['Nx']
data_param_Coreg['C'] =  [19]
data_param_Coreg['approach_type'] = 'CoCa_ICM'
data_param_Coreg['Var_unsc'] = list(np.arange(0, data_param_Coreg['Nc']))+ [data_param_Coreg['nDim']]
data_param_Coreg['Noutput'] = 2
data_param_Coreg['W_rank'] = 2
data_param_Coreg['BiasTL'] = True
data_param_Coreg['SampleSystem'] = False
data_param_Coreg['SystemID'] = 1

In [5]:
data = data_modified_np['KPhaffii_HSA']
result = result_modified_np['KPhaffii_HSA']
key = 'Coreg'
data_mod = dataprep(data, data_param_Coreg, key)

In [6]:
# random_seed = 37
# background_model_parameter = 'pre_ModelBackground_Coreg.pkl'
# Exp0 = design_initial_experiments(data_param_Coreg, initN, 'dummy', 
#                                   random_seed, background_model_parameter)

In [7]:
method = 'constant_liar'
batch_size = 11
output_file_name = './Round0/Round0_Coreg'
background_model_parameter = 'pre_ModelBackground_Coreg.pkl'
new_model_parameter = './Round0/0_ModelBackground_Coreg.pkl'

z_next, gp_actual = design_experiments(data_mod, result,batch_size, method,background_model_parameter, 
                            new_model_parameter , output_file_name)

(84, 5)
  ICM.                                           |               value  |  constraints  |  priors
  sum.MixtureViaSumAndProduct.mix                |  0.6282720539625114  |      +ve      |        
  sum.MixtureViaSumAndProduct.Mat52.lengthscale  |                (3,)  |      +ve      |        
  sum.white.variance                             |                 0.1  |   +ve fixed   |        
  B.W                                            |              (2, 2)  |               |        
  B.kappa                                        |                (2,)  |      +ve      |        
0
(5000, 4)
1
[13.          0.6263688   0.97849917  0.6675206   1.        ]
1
(5000, 4)
1
[1.50000000e+01 5.61514491e-03 5.87083037e-02 7.44476320e-01
 1.00000000e+00]
2
(5000, 4)
1
[12.          0.9825815   0.02725997  0.86944886  1.        ]
3
(5000, 4)
1
[18.          0.3282661   0.93724587  0.9189985   1.        ]
4
(5000, 4)
1
[1.10000000e+01 4.30705165e-03 8.95946723e-02 2.51425432e-01
 1.000000

In [ ]:
Carbon_Names[17]